# Document ingestion and audit

Run this first. Set SOURCE to your own file, list or directory. The default creates a synthetic Spanish corpus in a dedicated output directory; no external data required.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import markovify
from markovify.nlp import Corpus
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
OUT = ROOT / "notebooks" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
SEED = 42
print(markovify.__file__)

In [ ]:
from markovify.nlp import discover_files, supported_formats
demo = OUT / "demo"
demo.mkdir(exist_ok=True)
subjects = ["La proteína", "El receptor", "El ligando", "La enzima"]
verbs = ["interactúa con", "se une a", "modifica"]
objects = ["la proteína", "el receptor", "el ligando", "la membrana"]
contexts = ["durante el ensayo.", "en el sistema.", "bajo estas condiciones."]
sentences = [f"{s} {v} {o} {c}" for s in subjects for v in verbs for o in objects for c in contexts]
(demo / "sample.txt").write_text("\n".join(sentences), encoding="utf-8")
(demo / "sample.html").write_text("<p>La proteína interactúa con el receptor durante el ensayo.</p>", encoding="utf-8")
SOURCE = demo
FORMATS = None
print(supported_formats())
print(list(discover_files(SOURCE, formats=FORMATS)))

In [ ]:
raw = Corpus.load(SOURCE, formats=FORMATS, errors="skip")
report = pd.DataFrame(raw.load_report)
display(report)
if not len(raw):
    raise ValueError("No entries extracted")
raw.save(OUT / "raw.jsonl")
report.to_csv(OUT / "extraction_report.csv", index=False)
pd.DataFrame({"source": raw.sources, "text": raw.texts}).to_json(OUT / "raw_sources.jsonl", orient="records", lines=True, force_ascii=False)
report.groupby("status")["entries"].sum().plot.bar(title="Extracted entries by file status")
plt.ylabel("Entries")
plt.show()